In [ ]:
./Examples/Stereo/stereo_carla Vocabulary/ORBvoc.txt Examples/Stereo/AirSim.yaml /home/artur/Downloads/AirSim_runs/default  res.txt

In [1]:
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import subprocess
import os
import yaml
from typing import List, Optional
from tqdm.auto import tqdm

class SLAM:
    """Интерфейс для запуска и управления ORB-SLAM"""
    
    def __init__(self, orb_path: Optional[str] = None, voc_path: Optional[str] = None):
        """
        Args:
            orb_path: Путь к исполняемому файлу ORB-SLAM (если не указан, берется из .env)
            voc_path: Путь к словарю (если не указан, берется из .env)
        """
        self.orb_path = orb_path or os.getenv('ORB_SLAM_PATH')
        self.voc_path = voc_path or os.getenv('ORB_SLAM_VOC')
        
        if not self.orb_path or not os.path.exists(self.orb_path):
            raise FileNotFoundError(
                f"Исполняемый файл не найден: {self.orb_path}. "
                "Укажите путь в .env (ORB_SLAM_PATH) или передайте в конструктор."
            )
        
        if not self.voc_path or not os.path.exists(self.voc_path):
            raise FileNotFoundError(
                f"Словарь не найден: {self.voc_path}. "
                "Укажите путь в .env (ORB_SLAM_VOC) или передайте в конструктор."
            )
    
    def run_single(
        self,
        yaml_path: str,
        dataset_path: str,
        output_path: str
    ) -> None:
        """
        Запуск алгоритма на заданном наборе данных
        
        Args:
            yaml_path: Путь к конфигурационному файлу
            dataset_path: Путь к набору данных
            output_path: Путь для сохранения результатов
            
        Raises:
            RuntimeError: Если произошла ошибка при запуске или выполнении
        """
        # Проверяем существование файлов
        if not os.path.exists(yaml_path):
            raise RuntimeError(f"Не найден конфигурационный файл: {yaml_path}")
        if not os.path.exists(dataset_path):
            raise RuntimeError(f"Не найден набор данных: {dataset_path}")
        
        # Создаем директорию для результатов
        config_name = os.path.splitext(os.path.basename(yaml_path))[0]
        dataset_name = os.path.basename(dataset_path)
        result_dir = os.path.join(output_path, dataset_name, config_name)
        os.makedirs(result_dir, exist_ok=True)
        
        # Пути к файлам результатов
        trajectory_path = os.path.join(result_dir, 'trajectory.txt')
        if os.path.exists(trajectory_path):
            print(f"Файл {trajectory_path} уже существует")
            return
        log_path = os.path.join(result_dir, 'log.txt')
        
        try:
            # Запускаем процесс
            with open(log_path, 'w') as log_file:
                process = subprocess.Popen(
                    [
                        self.orb_path,
                        self.voc_path,
                        yaml_path,
                        dataset_path,
                        trajectory_path
                    ],
                    stdout=log_file,
                    stderr=subprocess.STDOUT,
                    text=True
                )
                returncode = process.wait()
                
                if returncode != 0:
                    raise RuntimeError(
                        f"Процесс ORB-SLAM завершился с ошибкой (код {returncode}). "
                        f"Подробности в файле: {log_path}"
                    )
                
        except FileNotFoundError:
            raise RuntimeError(f"Не удалось найти исполняемый файл: {self.orb_path}")
        except PermissionError:
            raise RuntimeError(f"Нет прав на выполнение файла: {self.orb_path}")
        except subprocess.SubprocessError as e:
            raise RuntimeError(f"Ошибка при выполнении процесса ORB-SLAM: {str(e)}")
    
    def run_experiment(
        self,
        configs: List[str],
        datasets: List[str],
        output_path: str,
        max_workers: Optional[int] = None
    ) -> None:
        """
        Запуск эксперимента на всех конфигурациях и датасетах
        
        Args:
            configs: Список путей к конфигурационным файлам
            datasets: Список путей к наборам данных
            output_path: Путь для сохранения результатов
            max_workers: Максимальное количество параллельных процессов
        """
        total_runs = len(configs) * len(datasets)
        print(f"\nЗапуск {total_runs} процессов "
              f"({len(configs)} конфигураций × {len(datasets)} датасетов)")
        
        max_workers = max_workers or int(os.getenv('MAX_WORKERS', '6'))
        
        try:
            with ProcessPoolExecutor(max_workers=max_workers) as executor:
                futures = []
                
                # Создаем задачи для всех комбинаций
                for dataset_path in datasets:
                    for config_path in configs:
                        process_func = partial(
                            self.run_single,
                            yaml_path=config_path,
                            dataset_path=dataset_path,
                            output_path=output_path
                        )
                        futures.append(executor.submit(process_func))
                
                # Отображаем прогресс
                failed = []
                with tqdm(total=total_runs, desc="Выполнение эксперимента") as pbar:
                    for future in futures:
                        try:
                            future.result()
                        except Exception as e:
                            failed.append(str(e))
                        finally:
                            pbar.update(1)
                
                # Выводим ошибки
                if failed:
                    print("\nПроизошли ошибки при выполнении некоторых процессов:")
                    for error in failed:
                        print(f"- {error}")
        except KeyboardInterrupt:
            print("\nПрерывание выполнения...")
            executor.shutdown(wait=False)
            raise

In [2]:
slam = SLAM("/home/artur/ROBOTICS/slam_project/slam/Examples/Stereo/stereo_carla", "/home/artur/ROBOTICS/slam_project/slam/Vocabulary/ORBvoc.txt")

In [ ]:
slam.run_experiment(['/home/artur/ROBOTICS/slam_project/slam/Examples/Stereo/AirSim.yaml'], ['/home/artur/Downloads/AirSim_runs/' + i for i in os.listdir('/home/artur/Downloads/AirSim_runs/')], '/home/artur/ROBOTICS/slam_project/slam/results', max_workers=6)

In [3]:
slam.run_experiment(['/home/artur/ROBOTICS/slam_project/slam/Examples/Stereo/AirSim.yaml'], ['/home/artur/Downloads/AirSim_runs/' + i for i in os.listdir('/home/artur/Downloads/AirSim_runs/')], '/home/artur/ROBOTICS/slam_project/slam/result_default', max_workers=6)


Запуск 24 процессов (1 конфигураций × 24 датасетов)


Выполнение эксперимента:   0%|          | 0/24 [00:00<?, ?it/s]


Произошли ошибки при выполнении некоторых процессов:
- Процесс ORB-SLAM завершился с ошибкой (код -7). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/road_wetness_25/AirSim/log.txt
- Процесс ORB-SLAM завершился с ошибкой (код -11). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/road_leaves_50/AirSim/log.txt
- Процесс ORB-SLAM завершился с ошибкой (код -11). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/fog_25/AirSim/log.txt
- Процесс ORB-SLAM завершился с ошибкой (код -11). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/road_snow_25/AirSim/log.txt
- Процесс ORB-SLAM завершился с ошибкой (код -11). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/falling_leaves/AirSim/log.txt
- Процесс ORB-SLAM завершился с ошибкой (код -11). Подробности в файле: /home/artur/ROBOTICS/slam_project/slam/result_default/snow/AirSim/log.txt
- Процесс ORB-SLAM завершился 

In [4]:
import pandas as pd
import os

gt_fp = os.path.join("/home/artur/Downloads/AirSim_runs", "default", "camera_info.csv")
gt_df = pd.read_csv(gt_fp)
traj_gt = gt_df.values
traj_gt = traj_gt[:, [1, 2, 3]]
traj_gt = traj_gt - traj_gt[0]
traj_gt

array([[ 0.000000e+00,  0.000000e+00,  0.000000e+00],
       [ 0.000000e+00,  0.000000e+00,  1.200000e-05],
       [ 0.000000e+00,  0.000000e+00,  2.100000e-05],
       ...,
       [-1.500062e+00, -1.945981e+00, -2.640000e-04],
       [-8.237950e-01, -1.960639e+00, -2.630000e-04],
       [-1.473750e-01, -1.975296e+00, -2.620000e-04]])

In [5]:
gt_df.shape

(2514, 8)

In [7]:
import numpy as np

pred_fp = os.path.join("/home/artur/ROBOTICS/slam_project/slam/results/default/AirSim/trajectory.txt")

traj_pred = np.loadtxt(pred_fp)[:, [11, 3, 7]]

In [13]:
RES_HOME = "/home/artur/ROBOTICS/slam_project/slam/results"
REC_HOME = "/home/artur/Downloads/AirSim_runs"
results = []
for exp_name in os.listdir(RES_HOME):
    gt_fp = os.path.join(REC_HOME, exp_name, "camera_info.csv")
    gt_df = pd.read_csv(gt_fp)
    traj_gt = gt_df.values
    traj_gt = traj_gt[:, [1, 2, 3]]
    traj_gt = traj_gt - traj_gt[0]
    
    pred_fp = os.path.join(RES_HOME, exp_name, "AirSim", "trajectory.txt")
    traj_pred = np.loadtxt(pred_fp)[:, [11, 3, 7]]

    res = np.sqrt(np.mean(np.sum(np.array((traj_pred - traj_gt))[:, [0, 1]] ** 2, axis=1)))
    results.append({'name': exp_name, 'res': res})

results = pd.DataFrame(results).set_index('name')
results.sort_values(by='res', inplace=True)
results


,res
name,
road_leaves_25,0.501279
road_leaves_50,0.501765
rain,0.568403
falling_leaves,0.604365
road_snow_25,0.612784
road_snow_75,0.665717
road_snow_50,0.683667
fog_25,0.687926
road_wetness_25,0.689345


In [15]:
(results['res'] - results.loc['default', 'res']).sort_values(ascending=False)

name
fog_100             6.212056
fog_75              0.781104
dust_75             0.443773
dust_100            0.272312
default             0.000000
fog_50             -0.029340
road_snow_100      -0.052513
snow               -0.116118
road_wetness_100   -0.142370
road_leaves_75     -0.174554
dust_50            -0.194256
road_wetness_50    -0.211729
road_leaves_100    -0.272886
dust_25            -0.276910
road_wetness_75    -0.306404
road_wetness_25    -0.346530
fog_25             -0.347949
road_snow_50       -0.352208
road_snow_75       -0.370157
road_snow_25       -0.423090
falling_leaves     -0.431510
rain               -0.467471
road_leaves_50     -0.534110
road_leaves_25     -0.534595
Name: res, dtype: float64

In [10]:
RES_HOME = "/home/artur/ROBOTICS/slam_project/slam/result_default"
REC_HOME = "/home/artur/Downloads/AirSim_runs"
results = []
for exp_name in os.listdir(RES_HOME):
    gt_fp = os.path.join(REC_HOME, exp_name, "camera_info.csv")
    gt_df = pd.read_csv(gt_fp)
    traj_gt = gt_df.values
    traj_gt = traj_gt[:, [1, 2, 3]]
    traj_gt = traj_gt - traj_gt[0]
    
    pred_fp = os.path.join(RES_HOME, exp_name, "AirSim", "trajectory.txt")
    traj_pred = np.loadtxt(pred_fp)[:, [11, 3, 7]]

    res = np.sqrt(np.mean(np.sum(np.array((traj_pred - traj_gt))[:, [0, 1]] ** 2, axis=1)))
    results.append({'name': exp_name, 'res': res})

results = pd.DataFrame(results)
results.sort_values(by='res', inplace=True)
results


,name,res
11,road_wetness_75,0.474934
21,road_leaves_75,0.496588
1,road_leaves_50,0.525826
0,road_wetness_25,0.558504
20,road_leaves_100,0.574881
7,road_snow_75,0.585992
4,falling_leaves,0.659344
3,road_snow_25,0.680523
2,fog_25,0.696568
22,road_wetness_50,0.730293


In [25]:
traj_gt

array([[ 0.000000e+00,  0.000000e+00,  0.000000e+00],
       [ 0.000000e+00,  0.000000e+00,  1.200000e-05],
       [ 0.000000e+00,  0.000000e+00,  2.100000e-05],
       ...,
       [-1.500062e+00, -1.945981e+00, -2.640000e-04],
       [-8.237950e-01, -1.960639e+00, -2.630000e-04],
       [-1.473750e-01, -1.975296e+00, -2.620000e-04]])

In [18]:
traj_pred

array([[ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 1.45708600e-03, -9.02550000e-04,  1.22638100e-03],
       [ 2.10283000e-03, -1.01722500e-03,  2.65171100e-03],
       ...,
       [ 6.03651480e-02, -1.90564477e+00, -2.65327883e+00],
       [ 7.65491247e-01, -1.90764916e+00, -2.63935828e+00],
       [ 1.42322993e+00, -1.91260910e+00, -2.64275551e+00]])